# Character training data generator
This notebook generates 42x42px PNG images of characters for training data.

In [1]:
import string
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from utils import split_into_char_images

In [2]:
keys = list(string.ascii_lowercase) + [str(i) for i in range(10)]
char_dict = {k: [] for k in keys}
char_dir = 'chars_train_data/'
# Delete all files in the target folder
for filename in tqdm(os.listdir(char_dir), desc="Deleting old files"):
    file_path = os.path.join(char_dir, filename)
    if os.path.isfile(file_path) and filename.lower().endswith(".png"):
        os.remove(file_path)
        
os.makedirs(char_dir, exist_ok=True)

Deleting old files: 100%|██████████████████████████████████████████████████████| 44364/44364 [00:04<00:00, 9170.78it/s]


# Start character separation

In [1]:
wrong_count = 0
images_count = 0

# Loop through all png in train/
for (root,dirs,files) in os.walk('clean_train_data/',topdown=True):
    for file in tqdm(files, desc="Loading images"):
        if file.endswith('.png'):
            if file[1] == "_":
                os.remove(os.path.join(root, file))
                continue
            if len(file.split("-")[1].split(".png")[0]) > 1:
                continue
                
            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)
            #processed_img = preprocessing_remove_lines(img) # Remove lines
            char_images = split_into_char_images(img) # Split chars
            ground_truth = img_path.split("-")[0].split("/")[-1] # Get Captcha label

            # If wrong seperation count, do not add to chars_train_data
            images_count += 1
            if len(char_images) != len(ground_truth):
                wrong_count += 1
                continue

            # Correct seperated
            for i in range(len(ground_truth)):
                k = ground_truth[i]
                char_dict[k].append(char_images[i])

print(f"Total captcha count: {images_count}")
print(f"Wrong character sep count: {wrong_count}")
            

NameError: name 'os' is not defined

### Save to PNG training data

In [4]:
# Save each image to PNG
for k, img_list in char_dict.items():
    for idx, img in tqdm(enumerate(img_list), desc=k):
        filename = f"{k}_{idx:05d}.png"  # e.g., a_0000.png
        path = os.path.join(char_dir, filename)
        #processed_img = process_character_image(img)
        cv2.imwrite(path, img)



a: 1198it [00:00, 3313.50it/s]
b: 1261it [00:00, 3380.61it/s]
c: 1217it [00:00, 4487.35it/s]
d: 1274it [00:00, 3339.30it/s]
e: 1269it [00:00, 2460.41it/s]
f: 1241it [00:00, 3734.96it/s]
g: 1263it [00:00, 4569.81it/s]
h: 1246it [00:00, 4415.56it/s]
i: 1238it [00:00, 5107.27it/s]
j: 1180it [00:00, 4382.77it/s]
k: 1241it [00:00, 3829.97it/s]
l: 1235it [00:00, 4107.29it/s]
m: 1245it [00:00, 3023.35it/s]
n: 1293it [00:00, 2040.20it/s]
o: 1221it [00:00, 3412.54it/s]
p: 1267it [00:00, 2527.09it/s]
q: 1286it [00:00, 5112.44it/s]
r: 1236it [00:00, 3682.87it/s]
s: 1222it [00:00, 3300.87it/s]
t: 1238it [00:00, 4432.83it/s]
u: 1207it [00:00, 3740.60it/s]
v: 1240it [00:00, 3933.65it/s]
w: 1209it [00:00, 4039.13it/s]
x: 1273it [00:00, 1883.33it/s]
y: 1188it [00:00, 4131.03it/s]
z: 1229it [00:00, 4057.04it/s]
0: 1243it [00:00, 4418.04it/s]
1: 1250it [00:00, 3540.28it/s]
2: 1173it [00:00, 3424.09it/s]
3: 1254it [00:00, 3727.24it/s]
4: 1233it [00:00, 3763.91it/s]
5: 1191it [00:00, 3399.37it/s]
6: 1224i

### Process character image after separation (unused)

In [22]:
def process_character_image(img):
    # --- Convert to grayscale ---
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # --- Invert so character = white (255), background = black (0) ---
    gray = 255 - gray
    
    # --- Threshold to make binary ---
    # _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # --- Find bounding box of character ---
    coords = cv2.findNonZero(gray)
    x, y, w, h = cv2.boundingRect(coords)
    
    # --- Crop + 3px padding ---
    pad = 3
    x1 = max(x - pad, 0)
    y1 = max(y - pad, 0)
    x2 = min(x + w + pad, gray.shape[1])
    y2 = min(y + h + pad, gray.shape[0])
    cropped = gray[y1:y2, x1:x2]

    # --- Normalize brightness: stretch intensity range to full 0–255 ---
    min_val, max_val = np.min(cropped), np.max(cropped)
    if max_val > min_val:  # avoid divide-by-zero if image is uniform
        cropped = (cropped - min_val) * (255.0 / (max_val - min_val))
        cropped = np.clip(cropped, 0, 255).astype(np.uint8)
    
    # --- Target canvas and padding ---
    target_size = 42
    pad = 3
    available_size = target_size - 2 * pad  # 38×38 drawable area

    h, w = cropped.shape
    scale = min(available_size / h, available_size / w)  # scale to fit within 38×38

    # --- Resize with preserved aspect ratio ---
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(cropped, (new_w, new_h), interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC)

    # --- Create blank 42×42 black background ---
    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    
    # --- Center the character ---
    y_offset = (target_size - new_h) // 2
    x_offset = (target_size - new_w) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized

    return canvas